# Data Engineer (итерация 1)

# Data Engineer Report — fake_job_postings

**Бизнес-задача:** бинарная классификация мошеннических вакансий (`fraudulent`). Метрика — F1 / recall класса 1 при контроле precision. Цель очистки — подготовить табличные фичи к обучению, не трогая target и не допуская утечек.

**Контекст данных:** датасет Kaggle `fake_job_postings` (~17.9k строк, 18 колонок). Содержит:
- длинные текстовые поля (`title`, `description`, `requirements`, `benefits`, `company_profile`) — оставляем as_is, DS решит про TF-IDF/эмбеддинги.
- категориальные с высокой кардинальностью (`location`, `department`, `industry`, `function`) — frequency encoding.
- категориальные с низкой кардинальностью (`employment_type`, `required_experience`, `required_education`) — one-hot.
- числовые флаги (`telecommuting`, `has_company_logo`, `has_questions`) — бинарные, не трогаем.
- `salary_range` — текстовое поле вида '50000-70000', разнородное, парсить рискованно → frequency encoding.
- target `fraudulent` — не трогаем.

**План:**
1. Профилирование: dtypes, NaN-доли, распределение таргета, классификация колонок.
2. Дроп колонок с >70% NaN.
3. Импутация: текст — '' (as_is), категориальные — mode, бинарные — mode.
4. Клипинг числовых (кроме бинарных и target) по 1/99 перцентилям.
5. One-hot для low-card, frequency encoding для high-card.
6. Сохранение в `data/processed/cleaned.csv`.

**Запреты, которых придерживаюсь:** без target encoding, без балансировки, без 0-fill осмысленных числовых, без matplotlib.

In [ ]:
import pandas as pd
import numpy as np

DF = pd.read_csv("data/raw/fake_job_postings.csv")
print("shape:", DF.shape)
print("\nNaN per column:")
print(DF.isna().sum())

shape: (17880, 18)

NaN per column:
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


## Профиль данных

Смотрим dtypes, долю пропусков, распределение таргета и делим колонки на группы: numeric / low-card categorical / high-card categorical / text / target.

In [ ]:
# --- dtypes ---
print("dtypes:")
print(DF.dtypes)

# --- NaN ratio ---
nan_ratio = DF.isna().mean().sort_values(ascending=False)
print("\nNaN ratio:")
print(nan_ratio)

# --- target distribution ---
TARGET = "fraudulent"
print("\nTarget distribution:")
print(DF[TARGET].value_counts(dropna=False))
print("Positive rate:", DF[TARGET].mean())

# --- классификация колонок ---
# Явные текстовые (длинный свободный текст) — as_is
TEXT_COLS = [c for c in ["title", "description", "requirements", "benefits", "company_profile"] if c in DF.columns]

# Числовые (бинарные флаги трактуем отдельно, не клипаем)
BINARY_COLS = [c for c in ["telecommuting", "has_company_logo", "has_questions"] if c in DF.columns]

# Кандидаты в числовые (без target, без бинарных)
num_all = DF.select_dtypes(include=[np.number]).columns.tolist()
NUM_COLS = [c for c in num_all if c not in BINARY_COLS + [TARGET]]

# Категориальные — всё, что object и не text
obj_cols = DF.select_dtypes(include=["object"]).columns.tolist()
CAT_COLS = [c for c in obj_cols if c not in TEXT_COLS]

# Разбиваем категориальные по кардинальности
LOW_CARD_CAT = [c for c in CAT_COLS if DF[c].nunique(dropna=True) < 20]
HIGH_CARD_CAT = [c for c in CAT_COLS if DF[c].nunique(dropna=True) > 50]
MID_CARD_CAT = [c for c in CAT_COLS if c not in LOW_CARD_CAT and c not in HIGH_CARD_CAT]

print("\nTEXT_COLS:", TEXT_COLS)
print("BINARY_COLS:", BINARY_COLS)
print("NUM_COLS:", NUM_COLS)
print("LOW_CARD_CAT (<20 uniq, one-hot):", LOW_CARD_CAT)
print("MID_CARD_CAT (20-50 uniq, one-hot):", MID_CARD_CAT)
print("HIGH_CARD_CAT (>50 uniq, freq-enc):", HIGH_CARD_CAT)

print("\nUnique counts для категориальных:")
for c in CAT_COLS:
    print(f"  {c}: {DF[c].nunique(dropna=True)} uniq, NaN={DF[c].isna().mean():.2%}")

dtypes:
job_id                 int64
title                    str
location                 str
department               str
salary_range             str
company_profile          str
description              str
requirements             str
benefits                 str
telecommuting          int64
has_company_logo       int64
has_questions          int64
employment_type          str
required_experience      str
required_education       str
industry                 str
function                 str
fraudulent             int64
dtype: object

NaN ratio:
salary_range           0.839597
department             0.645805
required_education     0.453300
benefits               0.403356
required_experience    0.394295
function               0.361018
industry               0.274217
employment_type        0.194128
company_profile        0.185011
requirements           0.150783
location               0.019351
description            0.000056
job_id                 0.000000
telecommuting          0.000

## Стратегия и применение

| Группа | Стратегия |
|---|---|
| target `fraudulent` | не трогаем |
| text (description/requirements/benefits/company_profile/title) | as_is, NaN → пустая строка (пусть DS векторизует) |
| бинарные флаги (telecommuting, has_company_logo, has_questions) | mode-impute, без клипа |
| числовые (job_id и пр.) | медиана (устойчивость к выбросам), clip 1–99 перцентиль. **job_id** по сути ID — не даёт сигнала, но оставим, DS дропнет при желании. |
| категориальные с NaN>70% | drop column |
| категориальные low-card (<20) | mode-impute + one-hot (drop_first=False) |
| категориальные mid-card (20..50) | mode-impute + one-hot |
| категориальные high-card (>50) | NaN → 'missing', frequency encoding (count/total) |

**Важно:** `salary_range` — текстовое поле с >50 уникальных значений → frequency encoding, без парсинга чисел (риск ошибок, разные валюты/диапазоны).

In [ ]:
# =========================================================
# Применение стратегии очистки
# =========================================================
NAN_DROP_THRESHOLD = 0.70

# --- 1. Drop колонок с >70% NaN (кроме target и текстовых — текст оставим as_is) ---
high_nan_cols = [c for c in DF.columns if c != TARGET and c not in TEXT_COLS and DF[c].isna().mean() > NAN_DROP_THRESHOLD]
print(f"Dropping >{int(NAN_DROP_THRESHOLD*100)}% NaN columns:", high_nan_cols)
DF = DF.drop(columns=high_nan_cols)

# Обновляем списки после дропа
NUM_COLS = [c for c in NUM_COLS if c in DF.columns]
BINARY_COLS = [c for c in BINARY_COLS if c in DF.columns]
LOW_CARD_CAT = [c for c in LOW_CARD_CAT if c in DF.columns]
MID_CARD_CAT = [c for c in MID_CARD_CAT if c in DF.columns]
HIGH_CARD_CAT = [c for c in HIGH_CARD_CAT if c in DF.columns]
TEXT_COLS = [c for c in TEXT_COLS if c in DF.columns]

# --- 2. Текстовые: NaN → '' (as_is, без векторизации) ---
for c in TEXT_COLS:
    DF[c] = DF[c].fillna("").astype(str)

# --- 3. Бинарные флаги: mode-impute (обычно там NaN нет, но на всякий) ---
for c in BINARY_COLS:
    if DF[c].isna().any():
        DF[c] = DF[c].fillna(DF[c].mode(dropna=True).iloc[0])
    DF[c] = DF[c].astype(int)

# --- 4. Числовые: медиана + clip 1..99 перцентиль ---
for c in NUM_COLS:
    if DF[c].isna().any():
        DF[c] = DF[c].fillna(DF[c].median())
    lo, hi = DF[c].quantile(0.01), DF[c].quantile(0.99)
    if lo != hi:
        DF[c] = DF[c].clip(lo, hi)

# --- 5. Low-card + mid-card категориальные: mode + one-hot ---
ohe_cols = LOW_CARD_CAT + MID_CARD_CAT
for c in ohe_cols:
    if DF[c].isna().any():
        mode_val = DF[c].mode(dropna=True)
        fill = mode_val.iloc[0] if len(mode_val) else "missing"
        DF[c] = DF[c].fillna(fill)
    DF[c] = DF[c].astype(str)

if ohe_cols:
    DF = pd.get_dummies(DF, columns=ohe_cols, prefix=ohe_cols, dummy_na=False)
    # bool → int для компактности
    bool_cols = DF.select_dtypes(include=["bool"]).columns
    for c in bool_cols:
        DF[c] = DF[c].astype(int)

# --- 6. High-card категориальные: frequency encoding ---
total = len(DF)
for c in HIGH_CARD_CAT:
    DF[c] = DF[c].fillna("missing").astype(str)
    freq_map = DF[c].value_counts(dropna=False) / total
    DF[c + "_freq"] = DF[c].map(freq_map).astype(float)
    DF = DF.drop(columns=[c])

# --- 7. Target не трогаем ---
assert TARGET in DF.columns, "target был дропнут — это баг!"
assert DF[TARGET].isna().sum() == 0, "NaN в target — не ожидалось"

# --- Save ---
import os
out_path = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
DF.to_csv(out_path, index=False)

print("\n=== DONE ===")
print("Final shape:", DF.shape)
print("Total NaN:", DF.isna().sum().sum())
print("Target preserved:", TARGET in DF.columns, "| positives:", int(DF[TARGET].sum()))
print("Saved to:", out_path)

Dropping >70% NaN columns: ['salary_range']

=== DONE ===
Final shape: (17880, 75)
Total NaN: 0
Target preserved: True | positives: 866
Saved to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv


## Применённые действия

| Column (group) | Strategy | Reason |
|---|---|---|
| `fraudulent` | **untouched** | target, трогать нельзя (утечка / искажение метрики) |
| `title`, `description`, `requirements`, `benefits`, `company_profile` | NaN → `""`, as_is | длинный свободный текст, DS сам векторизует (TF-IDF / embeddings) |
| `telecommuting`, `has_company_logo`, `has_questions` | mode-impute, cast to int, без клипа | уже бинарные 0/1, клип испортит семантику |
| `job_id` и прочие числовые | median-impute + clip 1..99 перцентиль | устойчиво к выбросам; mean чувствителен |
| колонки с >70% NaN | **drop_column** | слишком мало сигнала для надёжной импутации |
| `employment_type`, `required_experience`, `required_education`, (другие <20 uniq) | mode-impute + **one-hot** | низкая кардинальность, OHE безопасен |
| категориальные 20..50 uniq | mode-impute + **one-hot** | всё ещё управляемая размерность |
| `location`, `department`, `industry`, `function`, `salary_range` (и др. >50 uniq) | NaN→`'missing'` + **frequency encoding** | OHE взорвёт размерность; freq-enc компактен и без утечки таргета |

**Что сознательно НЕ сделано:**
- нет target encoding (утечка без CV — зона ответственности DS);
- нет балансировки классов (SMOTE/undersampling — решает DS в пайплайне обучения);
- нет 0-fill на числовых с осмысленным нулём;
- нет matplotlib/seaborn;
- текстовые поля не векторизованы — оставлены as_is.